# D4Explainer — Colab Setup
**Runtime:** GPU **A100 (40 GB)** required for paper-faithful runs — set via *Runtime > Change runtime type*.

T4 (16 GB) OOMs on every dataset under the paper-faithful loop because the Powerful net's `[bsz × sigma_length, N, N, hidden × num_layers]` activation tensor for one `backward()` exceeds 14 GB even at paper bsz (empirically confirmed on BA_shapes, bsz=4, padded N≈500, ~15 GB peak). Switching to A100 is a compute fix that preserves the paper's training procedure exactly; falling back to `--memory_efficient` on T4 is also possible but introduces a documented CF-loss approximation.

In [ ]:
# Cell 1: Upload and unzip code
from google.colab import files
import zipfile, os

print('Select D4Explainer.zip when the picker opens...')
uploaded = files.upload()  # select D4Explainer.zip

with zipfile.ZipFile('D4Explainer.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content/D4Explainer')
print('CWD:', os.getcwd())

In [ ]:
# Cell 2: Upload and unzip datasets
# Upload one zip per dataset — repeat for each: MUTAG, BA3, BA_shapes, etc.
from google.colab import files
import zipfile, os

os.makedirs('/content/D4Explainer/data', exist_ok=True)

print('Select dataset zip(s) when the picker opens...')
uploaded = files.upload()  # can select multiple files

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/D4Explainer/data/')
        print(f'Extracted {fname}')

print('\ndata/ contents:', os.listdir('/content/D4Explainer/data'))

In [ ]:
# Cell 3: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print('Device:', name)
    if 'A100' not in name:
        print(f'WARNING: paper-faithful loop expects A100 (40 GB). Current device "{name}" '
              f'will likely OOM on the BA_shapes smoke test (~15 GB peak activations).')
else:
    print('No GPU — check Runtime > Change runtime type > A100 GPU')

In [ ]:
# Cell 3b: Prevent TensorFlow GPU pre-allocation and verify clean GPU state
# TF is installed in Colab alongside PyTorch and grabs all GPU memory by default.
# Must run this before any training — if GPU memory looks wrong, restart the runtime first.
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import torch
allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM : {total:.1f} GiB')
print(f'Allocated  : {allocated:.2f} GiB  (should be ~0 on a fresh runtime)')
print(f'Reserved   : {reserved:.2f} GiB')
if allocated > 1.0:
    print('WARNING: >1 GiB already allocated — restart the runtime to clear leftover memory.')

In [ ]:
# Cell 4: Install dependencies
# Note: torch-scatter/sparse/cluster/spline-conv are pre-installed on modern Colab runtimes.
# torch-geometric version pin dropped — 2.4.0 has no wheel for Python 3.12+.
# pandas/networkx version pins dropped — pinned versions predate Python 3.12.
# rdkit-pypi renamed to rdkit.
import torch, subprocess

TORCH_VERSION = torch.__version__.split('+')[0]
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_str = result.stdout.split('release ')[-1].split(',')[0].strip().replace('.', '')
CUDA_TAG = f'cu{cuda_str}'
print(f'Torch: {TORCH_VERSION}, CUDA: {CUDA_TAG}')

# PyG sparse extensions (likely pre-installed; no-op if already present)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_TAG}.html

!pip install -q torch-geometric
!pip install -q networkx pandas rdkit pyemd

In [ ]:
# Cell 5: Smoke-test imports
import torch_geometric
import networkx, pandas, numpy
print('PyTorch:   ', torch.__version__)
print('PyG:       ', torch_geometric.__version__)
print('NetworkX:  ', networkx.__version__)
print('pandas:    ', pandas.__version__)
print('numpy:     ', numpy.__version__)
print('All imports OK')

In [ ]:
# Cell 6: Verify trained GNN checkpoints exist for every dataset.
import os
ckpts = [
    'param/gnns/BA_shapes_gcn.pt',
    'param/gnns/Tree_Cycle_gcn.pt',
    'param/gnns/Tree_Grids_gcn.pt',
    'param/gnns/cornell_gcn.pt',
    'param/gnns/NCI1_gcn.pt',
    'param/gnns/bbbp_gcn.pt',
    'param/gnns/ba3_gcn.pt',
    'param/gnns/mutag_gcn.pt',
]
missing = [c for c in ckpts if not os.path.exists(c)]
assert not missing, f'Missing checkpoints: {missing} — include them in the zip or train via gnns/'
print(f'All {len(ckpts)} GNN checkpoints found.')


In [ ]:
# Cell 6b: Mount Google Drive and symlink results/ -> Drive so run outputs survive runtime resets.
# All training cells write to results/{dataset}/{run_name}/{config.json,metrics.jsonl,best_model.pth};
# the symlink redirects those writes to Drive transparently — no flag changes needed below.
#
# Note: the upload zip excludes results/ contents (*/results/*) but the empty results/ directory
# itself is unzipped onto Colab. We detect that case and replace the empty dir with the symlink.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_RESULTS = '/content/drive/MyDrive/D4Explainer_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

local = '/content/D4Explainer/results'

if os.path.islink(local):
    # Already a symlink — re-point in case the target changed across sessions.
    os.unlink(local)
    os.symlink(DRIVE_RESULTS, local)
    print(f'Re-symlinked {local} -> {DRIVE_RESULTS}')
elif os.path.isdir(local):
    contents = os.listdir(local)
    if len(contents) == 0:
        os.rmdir(local)
        os.symlink(DRIVE_RESULTS, local)
        print(f'Removed empty real dir at {local} and symlinked -> {DRIVE_RESULTS}')
    else:
        raise RuntimeError(
            f'{local} exists as a real directory with contents {contents}. '
            f'Move them to {DRIVE_RESULTS} manually (e.g. shutil.copytree), then re-run this cell. '
            f'Refusing to silently send writes to ephemeral local disk.'
        )
elif not os.path.exists(local):
    os.symlink(DRIVE_RESULTS, local)
    print(f'Symlinked {local} -> {DRIVE_RESULTS}')

# Verify the write actually lands in Drive (the prior version of this cell printed a false-positive
# success message when the symlink was skipped — writes went to local disk silently).
test_path = os.path.join(local, '.write_test')
drive_test = os.path.join(DRIVE_RESULTS, '.write_test')
with open(test_path, 'w') as f:
    f.write('ok')
assert os.path.exists(drive_test), (
    f'WRITE LANDED ON LOCAL DISK, NOT DRIVE: {test_path} exists but {drive_test} does not. '
    f'Symlink is not redirecting writes — investigate before training.'
)
os.remove(test_path)
print(f'Drive write test verified end-to-end. Run outputs will persist to: {DRIVE_RESULTS}')


## Deviations from the original D4Explainer paper

The **default** training loop is now **paper-faithful** — single batched-sigma forward, one `loss.backward()`, CF loss averaged exactly over all 10 sigmas (matches commit `2cead28`). All memory-workaround changes below are now opt-in behind `--memory_efficient`.

### Hardware requirement (empirical, 2026-04-26)

The paper-faithful loop requires **A100 (40 GB)**. T4 (16 GB) OOMs on every dataset tried (confirmed on BA_shapes at paper bsz=4, padded N≈500: peak `[bsz × sigma_length, N, N, hidden × num_layers] = [40, 498, 498, 384] ≈ 15 GB`). This is a **compute** issue, not a methodological one — preserving the paper's training procedure on T4 would require either `--memory_efficient` (CF-loss approximation, see below) or aggressive `--max_graph_size` filtering (data change). A100 is the clean fix.

### Opt-in memory-mitigation stack (`--memory_efficient` path only)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Per-sigma BCE loss** | `--memory_efficient` | **Exact** | Original: one forward over all 10 sigmas, one backward. Memory-efficient: 10 forward+backward, each BCE scaled by `1/sigma_length`. Sum of gradients is algebraically identical. `total_sigmas` kwarg on `loss_func_bce` preserves the `1/|Σ|` weight. |
| **CF loss: single-sigma approximation** | `--memory_efficient` | **Approximation** | Original: CF loss uses the mean of scores across all 10 sigmas as soft edge weights into the frozen GNN. Memory-efficient: CF loss uses only the last sigma's score. Same expected direction but higher-variance gradient estimate. May lower reported fidelity slightly vs paper numbers. |
| **Gradient checkpointing on Powerful** | `--gradient_checkpointing` (only applied under `--memory_efficient`) | **Exact (math); ~2× compute** | Recomputes the full `Powerful` forward during backward. No numerical change, just compute/memory tradeoff. |
| **Mixed-precision (fp16) training** | `--use_amp` (only applied under `--memory_efficient`) | **Approximation (numerical)** | ~2× memory reduction on activations. Small numerical drift vs fp32 baseline. |

### Data-path changes (work in either mode)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Reduced batch size** | `--train_batchsize` | **Exact (loss); different SGD noise** | Changing bsz changes SGD variance and effective LR. Paper Table 7 bsz per-dataset is documented in README. |
| **Size-bucketed batch sampler** | `--size_bucketed` | **Exact (loss); different SGD sampling** | Groups graphs by node count so each batch pads to a tight N. Changes batch composition but loss per sample is unchanged. Useful for Mutagenicity if you bump bsz above the paper's 2. |
| **Max-graph-size filter** | `--max_graph_size` | **Data change** | Drops training graphs above a node-count threshold (~0.3% of Mutagenicity at N>100). Training-set reduction, not a loss change. Document the kept-fraction when reporting results. |
| **--debug_shapes** | `--debug_shapes` | N/A (diagnostic only) | Prints padded N and CUDA memory for the first 5 batches. No training effect. |

### Dataset naming caveat
The repo's `--dataset mutag` flag loads **Mutagenicity** (4337 graphs, max ~417 nodes), *not* classic TU MUTAG (188 graphs, max 28). The D4Explainer authors preserved this imprecise label; both the code and the paper tables refer to "MUTAG" but train on Mutagenicity.

### Recommended baseline-vs-extension protocol
Default (paper-faithful, A100):
- No `--memory_efficient`, no `--use_amp`, no `--gradient_checkpointing`
- Paper Table 7 hyperparameters per-dataset (bsz + alpha_cf; see README)

Then run the natural-gradient extension with the **same** flags + `--natural_gradient`. Single-flag toggle keeps the comparison clean.

If A100 is unavailable and you must use T4, fall back to `--memory_efficient` — matched on both baseline and extension. Document this as a compute fallback that introduces the CF-loss single-sigma approximation noted above.

---

### Extension (NOT paper — novel contribution)

| Change | Flag | Fidelity | Notes |
|---|---|---|---|
| **Natural-gradient hook on edge probabilities** | `--natural_gradient` | **Extension** | Registers a backward hook on each `score_batch` tensor that rescales `dL/d(score)` by the diagonal Fisher-Rao inverse metric `I(θ)^{-1} = θ(1-θ)`. Per CONTEXT.md §6 Alg 2: each score tensor is a point on the Bernoulli-product manifold `M_G`, so the natural-gradient correction is element-wise — O(|E|), not O(|E|²). Applied to all forward passes in both training paths. No change to loss, architecture, or data. |
| **Natural-gradient boundary clamp** | `--nat_grad_eps` (default 1e-6) | **Extension hyperparameter** | θ is clamped to `[eps, 1-eps]` inside the hook to avoid vanishing/exploding gradient at the manifold boundary. Rule-of-thumb default; tune if saturation is observed. |

**Experimental protocol:**
- Baseline run: paper-faithful default (no memory flags, Table 7 hyperparameters)
- Extension run: **same flags + `--natural_gradient`** — single-flag toggle keeps the comparison clean
- With `--natural_gradient` off, the code path is byte-for-byte identical to baseline

### Logging (infrastructure only, no algorithmic effect)

Each training run now writes to `results/{dataset}/{run_name}/`:
- `config.json` — snapshot of all CLI args (once, at training start)
- `metrics.jsonl` — one JSON record per epoch with train metrics, plus test metrics on eval epochs
- `best_model.pth` — existing best-checkpoint behavior, unchanged

`--run_name` (optional): nests outputs under a named subfolder so baseline and extension runs don't clobber each other. Default (unset) preserves the original `results/{dataset}/best_model.pth` path.

In [ ]:
# Cell 7: Smoke tests (1 epoch) — verify the paper-faithful loop runs end-to-end.
# Uses the same BA_shapes hyperparameters as the paired run cell below.
#
# --max_graph_size 450 is a documented data change (drops ~30% of BA_shapes train graphs
# above N=450). Required because BA_shapes' train-N distribution extends to 634 nodes,
# and even on A100 (40 GB) the Powerful net's [bsz × sigma_length, N, N, hidden × num_layers]
# activation tensor plus backward intermediates exceed 40 GB at N≳517 (empirically confirmed).
# This is a compute/data fix, NOT a methodological one — loss + optimizer are unchanged.

# Baseline smoke
!python main.py --dataset BA_shapes --epoch 1 --verbose 1 \
    --n_hidden 64 --num_layers 6 --train_batchsize 4 --alpha_cf 0.005 \
    --max_graph_size 450 \
    --run_name smoke_baseline

# Extension smoke (Fisher-Rao natural gradient)
!python main.py --dataset BA_shapes --epoch 1 --verbose 1 \
    --n_hidden 64 --num_layers 6 --train_batchsize 4 --alpha_cf 0.005 \
    --max_graph_size 450 \
    --natural_gradient --run_name smoke_natgrad


## Paired baseline + natural-gradient runs

One cell per dataset, ordered cheapest to most expensive. Each cell runs the paper-faithful baseline and then the `--natural_gradient` extension with identical hyperparameters. Outputs land in `results/{dataset}/baseline/` and `results/{dataset}/natgrad/`; compare the two `metrics.jsonl` files once both finish.

Run cells top-to-bottom and stop whenever your compute budget runs out. `--epoch` defaults to 800 (paper uses 1500 for Mutagenicity — bump if you want closer-to-paper numbers).

In [ ]:
# BA_shapes — Node-class. Train-N distribution extends to 634 nodes; --max_graph_size 450
# drops the upper ~30% to keep peak A100 activation memory under ~26 GB (empirical OOM line
# is N≈517 on A100 40GB at paper bsz=4). Loss/optimizer unchanged — documented data change only.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 4  alpha_cf 0.005
# Baseline
!python main.py --dataset BA_shapes --n_hidden 64 --num_layers 6 --train_batchsize 4 --alpha_cf 0.005 --max_graph_size 450 --run_name baseline 2>&1 | tee results/BA_shapes_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset BA_shapes --n_hidden 64 --num_layers 6 --train_batchsize 4 --alpha_cf 0.005 --max_graph_size 450 --natural_gradient --run_name natgrad 2>&1 | tee results/BA_shapes_natgrad.log


In [ ]:
# Tree_Cycle — Node-class, small graph.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 32 alpha_cf 0.1
# Baseline
!python main.py --dataset Tree_Cycle --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 --run_name baseline 2>&1 | tee results/Tree_Cycle_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset Tree_Cycle --n_hidden 64 --num_layers 6 --train_batchsize 32 --alpha_cf 0.1 --natural_gradient --run_name natgrad 2>&1 | tee results/Tree_Cycle_natgrad.log


In [ ]:
# Tree_Grids — Node-class, deeper model (num_layers 8).
# Paper Table 7: n_hidden 128 num_layers 8 train_batchsize 32 alpha_cf 0.05
# Baseline
!python main.py --dataset Tree_Grids --n_hidden 128 --num_layers 8 --train_batchsize 32 --alpha_cf 0.05 --run_name baseline 2>&1 | tee results/Tree_Grids_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset Tree_Grids --n_hidden 128 --num_layers 8 --train_batchsize 32 --alpha_cf 0.05 --natural_gradient --run_name natgrad 2>&1 | tee results/Tree_Grids_natgrad.log


In [ ]:
# cornell — SKIPPED for current run plan (base GNN test acc only ~54%, well below paper Table 6).
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 4  alpha_cf 0.05
# Baseline
# !python main.py --dataset cornell --n_hidden 128 --num_layers 6 --train_batchsize 4 --alpha_cf 0.05 --run_name baseline 2>&1 | tee results/cornell_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --dataset cornell --n_hidden 128 --num_layers 6 --train_batchsize 4 --alpha_cf 0.05 --natural_gradient --run_name natgrad 2>&1 | tee results/cornell_natgrad.log


In [ ]:
# NCI1 — Graph-class, ~4k small molecules.
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 32 alpha_cf 0.01
# Baseline
!python main.py --dataset NCI1 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.01 --run_name baseline 2>&1 | tee results/NCI1_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset NCI1 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.01 --natural_gradient --run_name natgrad 2>&1 | tee results/NCI1_natgrad.log


In [ ]:
# bbbp — SKIPPED for current run plan (base GNN val acc at chance; weak base => weak explainer comparison).
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 16 alpha_cf 0.005
# Baseline
# !python main.py --dataset bbbp --n_hidden 128 --num_layers 6 --train_batchsize 16 --alpha_cf 0.005 --run_name baseline 2>&1 | tee results/bbbp_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
# !python main.py --dataset bbbp --n_hidden 128 --num_layers 6 --train_batchsize 16 --alpha_cf 0.005 --natural_gradient --run_name natgrad 2>&1 | tee results/bbbp_natgrad.log


In [ ]:
# ba3 — Graph-class, motif ground truth.
# Paper Table 7: n_hidden 128 num_layers 6 train_batchsize 32 alpha_cf 0.05
# Baseline
!python main.py --dataset ba3 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.05 --run_name baseline 2>&1 | tee results/ba3_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset ba3 --n_hidden 128 --num_layers 6 --train_batchsize 32 --alpha_cf 0.05 --natural_gradient --run_name natgrad 2>&1 | tee results/ba3_natgrad.log


In [ ]:
# mutag — Mutagenicity, SLOWEST: paper bsz=2, max N~417. Falls back to --memory_efficient if OOM.
# Paper Table 7: n_hidden 64  num_layers 6 train_batchsize 2  alpha_cf 0.001
# Baseline
!python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --run_name baseline 2>&1 | tee results/mutag_baseline.log

# Natural-gradient extension (single-flag toggle: --natural_gradient)
!python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --natural_gradient --run_name natgrad 2>&1 | tee results/mutag_natgrad.log

# If Mutag OOMs on your GPU (only expected on T4), re-run with memory-efficient fallback:
# !python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --memory_efficient --size_bucketed --max_graph_size 100 --run_name baseline_me 2>&1 | tee results/mutag_baseline_me.log
# !python main.py --dataset mutag --n_hidden 64 --num_layers 6 --train_batchsize 2 --alpha_cf 0.001 --memory_efficient --size_bucketed --max_graph_size 100 --natural_gradient --run_name natgrad_me 2>&1 | tee results/mutag_natgrad_me.log
